In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "4" 

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from verl.utils.dataset.rl_dataset import RLHFDataset, collate_fn
from torch.utils.data import DataLoader
from vllm import LLM, SamplingParams

/mnt/petrelfs/zhangshilin/anaconda3/envs/deepscaler/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-09-16 17:00:31,544	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [2]:
data_idx = 50

# model_path = "/mnt/petrelfs/zhangshilin/rlvr_div/checkpoints/div/baseline_Llama_8B_clp_0.2_0.28_no_entropy/actor/global_step_100"
model_path = "/mnt/petrelfs/zhangshilin/rlvr_div/checkpoints/div/rs_equ_100step/actor/global_step_350"
tokenizer = AutoTokenizer.from_pretrained(model_path)
torch.cuda.empty_cache()
base_model = LLM(
    model=model_path,
    tensor_parallel_size=1,  # 使用全部8张GPU
    gpu_memory_utilization=0.85,  # 可以设置更高的内存利用率
    dtype="auto"
)

INFO 09-16 17:00:38 config.py:1450] Downcasting torch.float32 to torch.float16.
INFO 09-16 17:00:38 llm_engine.py:174] Initializing an LLM engine (v0.5.4) with config: model='/mnt/petrelfs/zhangshilin/rlvr_div/checkpoints/div/rs_equ_100step/actor/global_step_350', speculative_config=None, tokenizer='/mnt/petrelfs/zhangshilin/rlvr_div/checkpoints/div/rs_equ_100step/actor/global_step_350', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, rope_scaling=None, rope_theta=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=16384, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='outlines'), observability_config=ObservabilityConfig(otlp_traces_endpoint=None), seed=0, served_model_name=/mnt/petrelfs/z

Loading safetensors checkpoint shards:   0% Completed | 0/7 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  14% Completed | 1/7 [00:01<00:08,  1.43s/it]
Loading safetensors checkpoint shards:  29% Completed | 2/7 [00:02<00:07,  1.48s/it]
Loading safetensors checkpoint shards:  43% Completed | 3/7 [00:04<00:05,  1.33s/it]
Loading safetensors checkpoint shards:  57% Completed | 4/7 [00:05<00:04,  1.37s/it]
Loading safetensors checkpoint shards:  71% Completed | 5/7 [00:07<00:02,  1.43s/it]
Loading safetensors checkpoint shards:  86% Completed | 6/7 [00:07<00:01,  1.19s/it]
Loading safetensors checkpoint shards: 100% Completed | 7/7 [00:09<00:00,  1.30s/it]
Loading safetensors checkpoint shards: 100% Completed | 7/7 [00:09<00:00,  1.33s/it]



INFO 09-16 17:00:48 model_runner.py:732] Loading model weights took 14.2448 GB
INFO 09-16 17:00:50 gpu_executor.py:102] # GPU blocks: 59353, # CPU blocks: 4681
INFO 09-16 17:00:53 model_runner.py:1024] Capturing the model for CUDA graphs. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI.
INFO 09-16 17:00:53 model_runner.py:1028] CUDA graphs can take additional 1~3 GiB memory per GPU. If you are running out of memory, consider decreasing `gpu_memory_utilization` or enforcing eager mode. You can also reduce the `max_num_seqs` as needed to decrease memory usage.
INFO 09-16 17:01:05 model_runner.py:1225] Graph capturing finished in 12 secs.


In [3]:
train_data_path = "dataset/openr1.parquet"
train_dataset = RLHFDataset(parquet_files=train_data_path,
                            tokenizer=tokenizer,
                            prompt_key='prompt',
                            max_prompt_length=1024,
                            filter_prompts=True,
                            return_raw_chat=False,
                            truncation='error')

train_dataloader = DataLoader(dataset=train_dataset,
                            batch_size=5,
                            shuffle=True,
                            drop_last=True,
                            collate_fn=collate_fn)
test_data_path = "/mnt/petrelfs/zhangshilin/rlvr_div/dataset/valid.mmlu_pro.parquet"
test_dataset = RLHFDataset(parquet_files=test_data_path,
                            tokenizer=tokenizer,
                            prompt_key='prompt',
                            max_prompt_length=1024,
                            filter_prompts=True,
                            return_raw_chat=False,
                            truncation='error')

test_dataloader = DataLoader(dataset=test_dataset,
                            batch_size=1,
                            shuffle=True,
                            drop_last=True,
                            collate_fn=collate_fn)

original dataset len: 45792
filter dataset len: 45764
original dataset len: 12032
filter dataset len: 12005


In [4]:
test_data = test_dataset[data_idx]
# print(test_data['reward_model'])
input_text = tokenizer.decode(test_data['input_ids'], skip_special_tokens=True)

In [5]:
sampling_params = SamplingParams(
    temperature=0.6,
    # top_p=1.0,
    max_tokens=8192,
)

prompts = [input_text] * 8
outputs = base_model.generate(prompts, sampling_params)

Processed prompts: 100%|██████████| 8/8 [00:08<00:00,  1.06s/it, est. speed input: 316.11 toks/s, output: 548.02 toks/s]


In [6]:
group_rollout = []
for output in outputs:
    # 提取生成的文本
    full_text = output.outputs[0].text
    # 只保留输入之后新生成的部分
    # generated_text = full_text[len(input_text):]
    generated_text = full_text
    group_rollout.append(generated_text)

In [7]:
print(test_data['reward_model'])
for i in range(len(group_rollout)):
    print(f"********{i}**********")
    print(group_rollout[i])  # 从列表中提取文本
    print("__________end_____________\n")

{'ground_truth': 'A', 'style': 'rule'}
********0**********
To find Mr. Torres's total income from the stock for the year, we need to calculate the income from both the quarterly dividends and the extra year-end dividend.

1. First, let's calculate the total income from the quarterly dividends:
   - The quarterly dividend per share is $1.20.
   - Since there are four quarters in a year, the annual dividend per share from the quarterly payments would be \(4 \times 1.20\).

   So, the annual dividend per share from the quarterly payments is:
   \[
   4 \times 1.20 = 4.80 \text{ dollars per share}
   \]

   Since Mr. Torres owns 350 shares, the total income from the quarterly dividends for the year is:
   \[
   350 \times 4.80
   \]

2. Next, let's calculate the total income from the extra year-end dividend:
   - The extra year-end dividend per share is $0.30.
   - Since he owns 350 shares, the total income from the extra year-end dividend is:
   \[
   350 \times 0.30
   \]

3. Now, we nee

In [8]:
def extract_first_letter_after_answer(text):
    """
    Extract the first letter after "ANSWER:" from the text.
    
    Args:
        text (str): The text to extract the letter from.
        
    Returns:
        str or None: The extracted letter, or None if no valid letter is found.
    """
    if "ANSWER:" not in text:
        return None
    
    # 分割文本获取"ANSWER:"后面的内容
    parts = text.split("ANSWER:")
    if len(parts) < 2:
        return None
    
    # 获取"ANSWER:"后的内容并去除前后空白
    answer_part = parts[1].strip()
    
    # 如果没有内容，返回None
    if not answer_part:
        return None
    
    # 获取第一个非空白字符
    for char in answer_part:
        if char.strip():
            # 如果是字母，返回它
            if char.isalpha():
                return char.upper()  # 统一转为大写
            # 如果是数字或其他符号，也返回它
            return char
    
    return None

print(extract_first_letter_after_answer(group_rollout[0]))

None


In [9]:
think_phrases = [
    "alternatively",
    "as a result",
    "assume",
    "because",
    "calculate",
    "check",
    "compute",
    "confirm",
    "consequently",
    "consider",
    "define",
    "determine",
    "does not work",
    "doesn't work",
    "error",
    "evaluate",
    "finally",
    "find",
    "first",
    "firstly",
    "for example",
    "for instance",
    "get",
    "given there",
    "hence",
    "however",
    "if",
    "makes sence",
    "next",
    "not correct",
    "not working",
    "now",
    "re-calculate",
    "re-evaluate",
    "recalculate",
    "reevaluate",
    "since",
    "so",
    "solved",
    "step",
    "still not",
    "summarize",
    "then",
    "thereby",
    "therefore",
    "thus",
    "to",
    "try",
    "verify",
    "wait"
]
import re
def extract_phrases_regex(text, phrases_to_extract):
    occurrences = []
    for phrase in phrases_to_extract:
        pattern = re.compile(re.escape(phrase), re.IGNORECASE)
        for match in pattern.finditer(text):
            occurrences.append((match.start(), text[match.start():match.end()]))
    occurrences.sort(key=lambda x: x[0])
    extracted_phrases = [item[1] for item in occurrences]
    return extracted_phrases 

test_text = group_rollout[1]
test_phrase = extract_phrases_regex(test_text, phrases_to_extract=think_phrases)
print(test_phrase)

['to', 'find', 'To', 'to', 'to', 'to', 'calculate', 'First', 'determine', 'To', 'Since', 'to', 'calculate', 'if', 'first', 'calculate', 'Then', 'So', 'to', 'Next', 'determine', 'to', 'calculate', 'if', 'So', 'to', 'To', 'find', 'to', 'To', 'To', 'to', 'get', 'So', 'to', 'determine', 'To', 'to', 'to', 'to', 'get', 'to', 'Thus', 'to', 'to']


In [10]:
def calculate_phrase_repetition_penalty(phrases, n=1, penalty_value=0.01):
    """
    计算短语序列的重复惩罚值
    
    参数:
    phrases - 短语序列列表
    n - n-gram大小，默认为1（单个短语）
    penalty_value - 惩罚值，默认为0.1
    
    返回:
    penalties - 每个位置的惩罚值数组
    total_penalty - 总惩罚值
    """
    if not phrases:
        return [], 0.0
    
    # 序列长度
    l = len(phrases)
    
    # 如果序列长度小于n-gram大小，则返回零惩罚
    if l < n:
        return [0] * l, 0.0
    
    # 初始化惩罚向量和已观察到的n-grams集合
    penalties = [0.0] * l
    observed_ngrams = set()
    
    # 对每个可能的n-gram位置进行迭代
    for j in range(l - n + 1):
        # 提取当前的n-gram
        current_ngram = tuple(phrases[j:j+n])
        
        # 如果当前n-gram已经在观察集合中，应用惩罚
        if current_ngram in observed_ngrams:
            for t in range(j, j + n):
                penalties[t] += penalty_value
        
        # 将当前n-gram添加到观察集合
        observed_ngrams.add(current_ngram)
    
    # 计算总惩罚值
    total_penalty = sum(penalties)
    
    return penalties, total_penalty

phrases = test_phrase
    
# 计算不同n-gram大小的惩罚
results = {}
for n in range(1, 4):
    penalties, total = calculate_phrase_repetition_penalty(phrases, n=n)
    results[f"n={n}"] = {
        "penalties": penalties,
        "total_penalty": total,
        "normalized_penalty": total / len(phrases) if len(phrases) > 0 else 0
    }

# 显示结果
for key, value in results.items():
    print(f"\n{key}:")
    # print(f"  Phrases: {phrases}")
    print(f"  Penalties: {[round(p, 2) for p in value['penalties']]}")
    print(f"  Total Penalty: {value['total_penalty']:.2f}")
    print(f"  Normalized Penalty: {value['normalized_penalty']:.2f}")
    
    # 高亮显示被惩罚的短语
    highlighted = []
    for i, (phrase, penalty) in enumerate(zip(phrases, value['penalties'])):
        if penalty > 0:
            highlighted.append(f"[{phrase}]*")
        else:
            highlighted.append(phrase)
    print(f"  Highlighted: {highlighted}")


n=1:
  Penalties: [0.0, 0.0, 0.0, 0.01, 0.01, 0.01, 0.0, 0.0, 0.0, 0.01, 0.0, 0.01, 0.01, 0.0, 0.0, 0.01, 0.0, 0.0, 0.01, 0.0, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.0, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.0, 0.01, 0.01]
  Total Penalty: 0.31
  Normalized Penalty: 0.01
  Highlighted: ['to', 'find', 'To', '[to]*', '[to]*', '[to]*', 'calculate', 'First', 'determine', '[To]*', 'Since', '[to]*', '[calculate]*', 'if', 'first', '[calculate]*', 'Then', 'So', '[to]*', 'Next', '[determine]*', '[to]*', '[calculate]*', '[if]*', '[So]*', '[to]*', '[To]*', '[find]*', '[to]*', '[To]*', '[To]*', '[to]*', 'get', '[So]*', '[to]*', '[determine]*', '[To]*', '[to]*', '[to]*', '[to]*', '[get]*', '[to]*', 'Thus', '[to]*', '[to]*']

n=2:
  Penalties: [0.0, 0.0, 0.0, 0.0, 0.01, 0.01, 0.0, 0.0, 0.0, 0.0, 0.0, 0.01, 0.01, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.01, 0.02, 0.01, 0.01, 0.01, 0.0, 0.0, 0.01, 0.01, 0.01, 0.01, 0.0, 0.01, 0.01, 0.01, 0.02, 0.02